# Day 2 Starter Kit: Complete the Grounding (no Neo4j version)

**Your dataset: Patient, prescription & clinician (code CLN).** A clinic's operations log: appointments, registration, consultations, lab and imaging, pharmacy, insurance billing, wards and discharge.

**The idea.** Retrieval finds records that *sound* similar. It cannot tell whether two records describe the *same event*. The graph can, **if it knows which parts of a record identify an event.** That knowledge is a list of regular expressions, and writing it well is your job.

**What is given and working:** retrieval, the baseline RAG, graph building (lookup tables) from a *pattern registry*, and the grounded answer step. The registry has **one pattern** (Patient ID).

**What you do**
0. **Plug in your own LLM.** Add your key from any provider and, if needed, edit the one `call_llm` function so the notebook runs on it. Nothing else in the notebook touches the provider.
1. Explore the data and decide which other patterns identify a single event (**Kind 1**) and which are incidental fields reused across unrelated events (**Kind 2**: a room number, an amount, a semester...).
2. Add your patterns to the registry, each with a reason. List the lookalikes you *rejected*, and why.
3. Run the public question pack against baseline and grounded. Explain every outcome.
4. Say where the baseline fails and your graph works, and **one failure you still have.**

**No database needed.** This version keeps the graph in plain Python lookup tables, already built for you, so there is no Neo4j to set up. Your work is the registry. The judging is identical to the Neo4j version.

**What you submit:** this notebook, run top to bottom on a fresh runtime. The judges read your registry, your outcomes table and your remaining-failure note, then run a hidden question pack against your graph.

> Linking on the wrong thing is as bad as linking on nothing: a pattern that links unrelated records makes the system *invent* connections. Add a pattern only if you can say why two records sharing it are the same event.

## 1. Setup

In [ ]:
%pip install -q sentence-transformers hnswlib openai python-dotenv

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import hnswlib
import os

def get_secret(name):
    """Reads a secret from Colab's Secrets manager (padlock icon in the left
    sidebar) if running on Colab; falls back to a local .env file otherwise."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.getenv(name)

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model ready")

### Your LLM provider

Bring your own key (any provider) and add it to Colab's Secrets manager (padlock icon) as `LLM_API_KEY`. Every LLM call in this notebook goes through the single function `call_llm` below.

Run the cell: it should print a one-word reply. If it errors, fix this cell before going on.

In [ ]:
# ---- YOUR LLM PROVIDER (your own key, any provider) ----
# Add your key to Colab Secrets as LLM_API_KEY. The whole notebook talks to the model through
# ONE function, call_llm(prompt) -> str, so this is the only cell you change for a new provider.
# The default below works for OpenAI and for any provider with an OpenAI-compatible endpoint:
# set LLM_MODEL (and LLM_BASE_URL if it is not OpenAI) from your provider's documentation.
# A provider with its own SDK: keep the name and signature of call_llm, rewrite the body.
from openai import OpenAI

LLM_MODEL = "gpt-4o-mini"
LLM_BASE_URL = None  # None = OpenAI. Otherwise your provider's OpenAI-compatible URL

client = OpenAI(api_key=get_secret("LLM_API_KEY"), base_url=LLM_BASE_URL)

def call_llm(prompt):
    """Sends one prompt, returns the model's reply as plain text.
    Keep temperature=0: otherwise the same question can give a different answer on each run."""
    response = client.chat.completions.create(
        model=LLM_MODEL, temperature=0,
        messages=[{"role": "user", "content": prompt}])
    return response.choices[0].message.content

print("LLM check:", call_llm("Reply with the single word: ready"))

### No database needed: the graph lives in lookup tables

This version of the starter holds the graph in plain Python lookup tables instead of a Neo4j database, so there is nothing to sign up for or connect. Section 7 builds the tables from your registry and `graph_check` in section 8 walks them. They answer the same question a graph database would: *is there a path of shared entities between these two records?*

## 2. The dataset: Patient, prescription & clinician

141 records in 12 categories. A clinic's operations log: appointments, registration, consultations, lab and imaging, pharmacy, insurance billing, wards and discharge. Four pairs of categories deliberately share vocabulary. Most records stand alone; some carry an identifier or a code.

In [ ]:
DATASET_ID = "CLN"
DATASET_NAME = "Patient, prescription & clinician"

rec_categories = {
    'Appointments': [
        'A follow-up appointment was booked for next week',
        "The clinic rescheduled a morning slot after a doctor's leave was approved",
        'An appointment reminder was sent by text message',
        'A walk-in visit was converted into a scheduled appointment',
        'Online booking was paused during a system upgrade',
        'A cancelled appointment slot was offered to the waiting list',
        'An interpreter was arranged for a scheduled visit',
        'A telehealth consultation link was sent to the visitor',
        'Appointment no-shows were reviewed at the weekly meeting',
        'Patient P2063 booked a follow-up appointment after the first visit',
        'Clinician DR12 approved a leave and the slot was rescheduled',
        'Wing B was reserved for the orthopaedic outpatient sessions',
    ],
    'Registration': [
        'A new visitor completed the registration form at the front desk',
        'An identity document was scanned and attached to the record',
        'A duplicate registration record was merged by the records team',
        "The front desk updated a visitor's contact details",
        'Self-service kiosks were added to the registration area',
        'A registration queue formed during the morning rush',
        'Consent forms were collected at registration',
        "A visitor's insurance details were captured at the desk",
        'The registration team corrected a misspelt name',
        'Patient P2044 registered at the front desk and completed the consent forms',
    ],
    'Consultations': [
        "The doctor reviewed the visitor's history before the consultation",
        'A specialist referral was written after the examination',
        'A second opinion was requested by the family',
        'The consultation notes were typed into the clinical system',
        'A dietitian advised on a meal plan during the visit',
        'Follow-up advice was shared in a printed summary',
        'A paediatric consultation ran over the scheduled time',
        'The doctor explained the test results in plain language',
        'A short consultation was held by phone',
        'Prescription RX114 was written after the examination',
        'Clinician DR07 held the consultation and wrote a specialist referral',
        'Prescription RX101 was revised by the doctor after the interaction was flagged',
        'Prescription RX122 was written for a 250 mg dose of an antibiotic',
        'A 500 mg antibiotic course was prescribed for a throat infection',
    ],
    'Lab_Tests': [
        'Blood samples were collected and labelled at the phlebotomy desk',
        'The lab analyser was recalibrated after a quality check',
        'A urine sample was rejected for being insufficient',
        "Test results were released to the doctor's inbox",
        'A critical value was phoned through to the ward',
        'The lab ran a batch of cholesterol panels',
        'A sample was recollected after a labelling error',
        'External reference lab results were received by courier',
        'Reagent stock was replenished after a shortage',
        'Sample LAB-3305 was collected and labelled at the phlebotomy desk',
        "Sample LAB-3305 results were released to the doctor's inbox",
        "Patient P2058's blood test was repeated after a labelling error",
    ],
    'Imaging': [
        'A chest X-ray was reported by the radiologist',
        'The MRI scanner was serviced by the vendor',
        'An ultrasound appointment was moved to the afternoon',
        'A CT scan was repeated because of patient movement',
        'Imaging reports were uploaded to the clinical system',
        'A radiographer completed a safety check on the equipment',
        'A contrast allergy was noted before a scan',
        'The imaging queue was reorganised for urgent cases',
        'An old film archive was digitised',
        'Clinician DR07 requested an urgent CT scan',
        'Patient P2058 had a chest X-ray reported by the radiologist',
    ],
    'Pharmacy': [
        'The pharmacy dispensed a repeat prescription',
        'A drug interaction was flagged by the pharmacist',
        'Pharmacy stock of antibiotics was replenished',
        'A generic substitute was offered for an unavailable brand',
        'Expired medicines were removed from the shelves',
        'Labels with dosage instructions were printed for a dispensing',
        'The pharmacy confirmed a prescription with the clinic',
        'Cold chain storage for vaccines was checked',
        'A pharmacist counselled a visitor on how to take a medicine',
        'Prescription RX114 was dispensed by the pharmacy',
        'Prescription RX101 was flagged by the pharmacist for a drug interaction',
        'Prescription RX122 was refilled at the pharmacy',
        'A 500 mg paracetamol tablet was dispensed to a walk-in visitor',
        'A batch of 250 mg tablets was recalled by the manufacturer',
    ],
    'Billing_Insurance': [
        "A consultation fee was posted to the visitor's account",
        'An insurance pre-authorisation was requested',
        'A claim was rejected for a missing form',
        'A co-payment was collected at the front desk',
        'The billing team corrected a coding error on a claim',
        'An insurer queried the length of a hospital stay',
        'A bill estimate was shared before a scheduled procedure',
        'A claim reimbursement was received from the insurer',
        'A payment plan was arranged for an outstanding bill',
        "Patient P2044's insurance claim was submitted after the consultation",
        'Patient P2077 was given a bill estimate before the procedure',
        'Billing queries peaked after the evening clinic',
    ],
    'Wards_Admissions': [
        'A bed was allocated to a person admitted through emergency',
        'Admission paperwork was completed at the bedside',
        'A transfer to the intensive care unit was arranged',
        'Bed occupancy was reviewed during the morning rounds',
        'A planned admission was postponed by the surgeon',
        'A relative was given visiting hour instructions',
        'A shortage of beds triggered a diversion request',
        'Ward rounds were completed before noon',
        'A person was moved to a single room for isolation',
        'Patient P2031 was admitted through emergency and allocated a bed',
        'Patient P2058 was moved out of Ward 3 to a single room',
        'A bed shortage was reported in Ward 6',
    ],
    'Discharge': [
        'A discharge summary was prepared for the family doctor',
        'Discharge was delayed while awaiting a pharmacy supply',
        'Home care instructions were explained to the family',
        'A follow-up visit was arranged at discharge',
        'Transport was booked for a person going home',
        'The discharge lounge was opened to free up beds',
        'A discharge against advice was documented',
        'The billing clearance was completed before discharge',
        'A nurse reviewed the discharge checklist with the family',
        'Patient P2063 was discharged with home care instructions',
    ],
    'Nursing': [
        'Nursing handover was completed at the change of shift',
        'Vital signs were recorded every four hours',
        'A fall-risk assessment was repeated for an elderly person',
        'A nurse administered scheduled medication',
        'Wound dressings were changed according to the care plan',
        'A nurse escalated a rise in temperature',
        'Nursing staffing was adjusted for the weekend',
        'Pressure sore prevention rounds were documented',
        'A nurse trained a family member to give injections',
        'Patient P2031 had a fall-risk assessment repeated',
        'Patient P2077 complained about noise in Ward 3 at night',
        'Extra nurses were rostered for the evening clinic',
    ],
    'Facilities': [
        'The waiting area chairs were replaced',
        'A leaking tap in the restroom was repaired',
        'The generator was tested before the monsoon',
        'Signage was updated for the new entrance',
        'The lift was out of service for repairs',
        'Cleaning contractors changed their schedule',
        'Biomedical engineering calibrated the monitors',
        'Parking for visitors was extended',
        'An air conditioner was serviced in the waiting hall',
        'The generator in Wing B failed during the monsoon storm',
    ],
    'Infection_Control': [
        'Hand hygiene audits were carried out on all floors',
        'A cluster of infections prompted extra cleaning',
        'Isolation signs were placed outside a room',
        'Surgical instruments were sterilised in the central unit',
        'Staff were reminded to wear masks in clinical areas',
        'Waste segregation was inspected by the infection control nurse',
        'A vaccine drive was run for staff',
        'A sterilisation cycle was repeated after a failed indicator',
        'Environmental swabs were sent to the lab',
        'Ward 6 was closed for deep cleaning after the infection cluster',
        'Clinician DR19 led the infection control audit',
        'Swab batch LAB-2201 was sent to the lab for culture',
    ],
}

rec_texts = []
rec_labels = []
for category, sentences in rec_categories.items():
    for s in sentences:
        rec_texts.append(s)
        rec_labels.append(category)

print(f"Dataset {DATASET_ID}: {len(rec_texts)} records, {len(rec_categories)} categories")

## 3. Retrieval (given: this part works)

In [ ]:
rec_embeddings = model.encode(rec_texts, show_progress_bar=True)
rec_embeddings = np.ascontiguousarray(rec_embeddings.astype('float32'))

dim = rec_embeddings.shape[1]
rec_index = hnswlib.Index(space='cosine', dim=dim)
rec_index.init_index(max_elements=len(rec_texts), ef_construction=200, M=16)
rec_index.add_items(rec_embeddings, np.arange(len(rec_texts)))
rec_index.set_ef(50)
print(f"Index built: {len(rec_texts)} records")


def retrieve(question, k=3):
    q_emb = model.encode([question])[0].astype('float32').reshape(1, -1)
    labels, distances = rec_index.knn_query(q_emb, k=k)
    return [(int(idx), rec_texts[idx], rec_labels[idx]) for idx in labels[0]]

## 4. Baseline RAG (given: no grounding)

The plain Week 2 pipeline. It retrieves similar records and trusts the LLM to reason about whether they are connected.

In [ ]:
# The plain Week 2 prompt -- no grounding. temperature=0 so runs are repeatable.
BASELINE_PROMPT = """Answer the question using the context below.

Context:
{context}

Question: {question}

Answer:"""

def baseline_rag_answer(question, k=3):
    results = retrieve(question, k=k)
    context = "\n".join(f"- {text}" for _, text, _ in results)
    prompt = BASELINE_PROMPT.format(context=context, question=question)
    return {"question": question, "retrieved": results, "answer": call_llm(prompt)}

## 5. Explore: what could identify one event?

Read the records below. For each string with digits, ask: *if I saw this same string in a different record, would it still point at the same event?*

In [ ]:
import re

def show_id_like_records(max_rows=80):
    """Prints every record containing a digit. Read them: which of these strings
    would still point at ONE specific event if you saw it in a different record?"""
    n = 0
    for i, t in enumerate(rec_texts):
        if re.search(r"\d", t):
            print(f"R{i:<3} {t}")
            n += 1
            if n >= max_rows:
                break
    print(f"\n{n} records shown")

show_id_like_records()

## 6. YOUR TASK: the pattern registry

Each entry becomes a graph node label. Two records that mention the *same identifier* get connected through that node. Edit this cell, then run it.

In [ ]:
import re

PATTERNS = [
    {
        "name": 'Patient',
        "regex": r"\bPatient\s+(P\d{4})\b",   # group 1 = the identifier
        "kind": 1,
        "example": 'Patient P2044',
        "why": 'A patient ID names one person: the registration and the insurance claim for P2044 are about the same patient.',
    },
    # YOUR PATTERNS GO HERE. One entry per pattern, same five fields as above.
    # Not sure a pattern is right? Test it first:
    #   re.findall(r"your regex", campus_texts[i])   or loop over all campus_texts.
    # {
    #     "name": "Prescription",                      # names the type of identifier
    #     "regex": r"...",                        # ONE capture group around the identifier
    #     "kind": 1,                              # 1 = names ONE event/person; 2 = incidental field
    #     "example": "...",                       # a string from the data it matches
    #     "why": "...",                           # why two records sharing it ARE the same event
    # },
]

REJECTED_CANDIDATES = [
    # Things that LOOK like identifiers but you decided NOT to link on. Credit is given for good reasons.
    # {"candidate": "...", "kind": 2, "why_rejected": "..."},
]

def validate_registry():
    """Checks that every pattern is complete and compiles. Run this before building the graph."""
    problems = []
    for p in PATTERNS:
        label = p.get("name", "?")
        for key in ("name", "regex", "kind", "example", "why"):
            if p.get(key) in (None, ""):
                problems.append(f"{label}: missing '{key}'")
        if p.get("name") and not re.fullmatch(r"[A-Za-z][A-Za-z0-9_]*", p["name"]):
            problems.append(f"{label}: name must be letters/digits/underscore only")
        try:
            rx = re.compile(p.get("regex", ""))
            if rx.groups < 1:
                problems.append(f"{label}: regex needs ONE capture group around the identifier")
        except re.error as e:
            problems.append(f"{label}: regex does not compile ({e})")
        if p.get("kind") not in (1, 2):
            problems.append(f"{label}: kind must be 1 or 2")
    return problems

def show_register():
    print(f"{'Pattern':<10} {'Kind':<5} {'Example':<20} Why")
    for p in PATTERNS:
        print(f"{p['name']:<10} {p['kind']:<5} {p['example']:<20} {p['why']}")
    if REJECTED_CANDIDATES:
        print("\nConsidered but rejected:")
        for r in REJECTED_CANDIDATES:
            print(f"  - {r['candidate']} (Kind {r['kind']}): {r['why_rejected']}")

problems = validate_registry()
print("Registry problems:", problems if problems else "none")
show_register()

## 7. Build the lookup tables from your registry (given)

Re-run this cell every time you change the registry. Three tables hold the graph: `mentions` (one row per record-entity match) and the same rows indexed both ways (`entities_of`, `records_of`). The summary shows how many links each pattern created: a pattern with **0 shared identifiers** adds nothing, and one that links *many* records deserves suspicion.

In [ ]:
def extract_entities(text):
    """Applies every registered pattern. Returns [(pattern_name, identifier), ...]."""
    found = []
    for p in PATTERNS:
        for m in re.finditer(p["regex"], text):
            ent = (p["name"], m.group(1))
            if ent not in found:
                found.append(ent)
    return found

assert not validate_registry(), "Fix the registry problems above first"

# Table 1: one row per mention: (record_id, entity_type, entity_value)
mentions = []
for i, text in enumerate(rec_texts):
    for etype, value in extract_entities(text):
        mentions.append((f"R{i}", etype, value))

# Tables 2 and 3: the same rows indexed both ways, so we can walk record -> entity -> record
entities_of = {}   # "R12" -> {("Student", "S001"), ...}
records_of = {}    # ("Student", "S001") -> {"R12", "R57", ...}
for rec, etype, value in mentions:
    entities_of.setdefault(rec, set()).add((etype, value))
    records_of.setdefault((etype, value), set()).add(rec)

print(f"Lookup tables built from your registry: {len(mentions)} mention rows")
for name in sorted({m[1] for m in mentions}):
    ids = {}
    for rec, etype, value in mentions:
        if etype == name:
            ids[value] = ids.get(value, 0) + 1
    shared = sum(1 for c in ids.values() if c > 1)
    print(f"  {name:<10} {sum(ids.values()):>3} mentions | {len(ids):>3} distinct identifiers | {shared} shared by 2+ records (= links)")

In [ ]:
# What the lookup table looks like: the first 12 rows
print(f"{'record':<8}{'entity_type':<14}entity_value")
for row in mentions[:12]:
    print(f"{row[0]:<8}{row[1]:<14}{row[2]}")

## 8. Grounded answer (given)

`graph_check` walks the lookup tables to see whether two records are connected (and through which entity). The grounded step asks it about each pair of retrieved records, then hands the LLM those facts with record IDs. `temperature=0`, so repeated runs give the same answer.

In [ ]:
from collections import deque

def graph_check(id_a, id_b, max_hops=6):
    """Are two records connected? Returns {"disconnected": bool, "via": [...]}.
    Walks record -> entity -> record using the lookup tables; each step is one hop,
    and the walk stops after 6 hops."""
    start, goal = ("R", id_a), ("R", id_b)
    prev, queue = {start: None}, deque([(start, 0)])
    while queue:
        node, depth = queue.popleft()
        if node == goal:
            break
        if depth >= max_hops:
            continue
        if node[0] == "R":
            neighbours = [("E", etype, value) for etype, value in entities_of.get(node[1], ())]
        else:
            neighbours = [("R", rec) for rec in records_of[(node[1], node[2])]]
        for nb in neighbours:
            if nb not in prev:
                prev[nb] = node
                queue.append((nb, depth + 1))
    if goal not in prev:
        return {"disconnected": True, "via": None}
    path, node = [], goal
    while node is not None:
        path.append(node)
        node = prev[node]
    return {"disconnected": False, "via": [f"{n[1]} {n[2]}" for n in reversed(path) if n[0] == "E"]}


GROUNDED_PROMPT = """You answer questions about outpatient clinic and hospital operations (appointments, labs, pharmacy, wards, billing) records. Use ONLY the records and graph facts below.

Rules:
1. The question's wording is a claim to check, NOT evidence. Do not accept a link just because the question assumes one.
2. Doses, ward numbers, wing letters, clinic sessions, dates and similar shared values are NOT links.
3. The graph facts are ground truth about links. If the two records the question is about are NOT CONNECTED,
   say plainly that no recorded link exists between them. If they are CONNECTED, say so and name what they share.
4. If the records do not answer the question, say so. Cite record IDs like [R12].

Records:
{context}

Graph facts (checked in the database):
{graph_facts}

Question: {question}

Answer:"""


def grounded_answer(question, k=3):
    results = retrieve(question, k=k)
    ids = [f"R{idx}" for idx, _, _ in results]

    fact_lines, verified_connect, verified_disconnect = [], False, False
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            chk = graph_check(ids[i], ids[j])
            if chk is None:
                continue
            if chk["disconnected"]:
                verified_disconnect = True
                fact_lines.append(f"- [{ids[i]}] and [{ids[j]}]: NOT CONNECTED -- no shared named entity links them.")
            else:
                verified_connect = True
                fact_lines.append(f"- [{ids[i]}] and [{ids[j]}]: CONNECTED -- via {', '.join(chk['via'] or [])}.")

    context = "\n".join(f"- [R{idx}] {text}" for idx, text, _ in results)
    prompt = GROUNDED_PROMPT.format(context=context,
                                    graph_facts="\n".join(fact_lines) or "(no pairs to check)",
                                    question=question)
    return {"question": question, "retrieved": results, "graph_facts": fact_lines,
            "verified_connect": verified_connect, "verified_disconnect": verified_disconnect,
            "answer": call_llm(prompt)}

## 9. The public question pack

8 adversarial questions (the right answer is *no recorded connection*) and 3 positive controls (the right answer is *yes*). The judges will run a **different, hidden** pack of the same shapes, so fix your **registry**, not individual questions.

In [ ]:
PUBLIC_ADVERSARIAL = [{'id': 'P1',
  'question': 'Was the walk-in who was converted to an appointment the same visitor who completed '
              'registration at the front desk?',
  'trap_records': ['A walk-in visit was converted into a scheduled appointment',
                   'A new visitor completed the registration form at the front desk'],
  'correct_answer_must_state': 'No stated connection: neither record names a patient ID, so they '
                               'cannot be shown to be the same visitor.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (visit)'},
 {'id': 'P2',
  'question': 'Were the cholesterol panels ordered because of the chest X-ray?',
  'trap_records': ['The lab ran a batch of cholesterol panels',
                   'A chest X-ray was reported by the radiologist'],
  'correct_answer_must_state': 'No stated connection: a lab batch and an imaging report name no '
                               'shared patient or sample.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (tests)'},
 {'id': 'P3',
  'question': 'Was the insurance claim rejected because a generic substitute was offered?',
  'trap_records': ['A generic substitute was offered for an unavailable brand',
                   'A claim was rejected for a missing form'],
  'correct_answer_must_state': 'No stated connection: the claim was rejected for a missing form, '
                               'and no patient links it to the substitute.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (claims)'},
 {'id': 'P4',
  'question': 'Did the delayed discharge cause the shortage of beds that triggered the diversion '
              'request?',
  'trap_records': ['Discharge was delayed while awaiting a pharmacy supply',
                   'A shortage of beds triggered a diversion request'],
  'correct_answer_must_state': 'No stated connection: neither record names a patient or ward.',
  'expect_connected': False,
  'kind': 'Vocabulary overlap (beds)'},
 {'id': 'P5',
  'question': 'Was the bed shortage in Ward 6 caused by the deep cleaning closure of Ward 6?',
  'trap_records': ['Ward 6 was closed for deep cleaning after the infection cluster',
                   'A bed shortage was reported in Ward 6'],
  'correct_answer_must_state': 'No stated connection: both mention Ward 6, but a ward is a place '
                               'used by many events, not an event identifier.',
  'expect_connected': False,
  'kind': 'Kind 2 trap: ward'},
 {'id': 'P6',
  'question': 'Did the orthopaedic outpatient sessions in Wing B cause the generator failure?',
  'trap_records': ['Wing B was reserved for the orthopaedic outpatient sessions',
                   'The generator in Wing B failed during the monsoon storm'],
  'correct_answer_must_state': 'No stated connection: sharing a wing name is not sharing an event.',
  'expect_connected': False,
  'kind': 'Kind 2 trap: wing'},
 {'id': 'P7',
  'question': 'Was the 500 mg paracetamol dispensed against the 500 mg antibiotic prescription?',
  'trap_records': ['A 500 mg paracetamol tablet was dispensed to a walk-in visitor',
                   'A 500 mg antibiotic course was prescribed for a throat infection'],
  'correct_answer_must_state': 'No stated connection: the strengths match by coincidence, and the '
                               'drugs and records differ.',
  'expect_connected': False,
  'kind': 'Kind 2 trap: dose'},
 {'id': 'P8',
  'question': 'Was a second opinion requested because the lift was out of service?',
  'trap_records': ['A second opinion was requested by the family',
                   'The lift was out of service for repairs'],
  'correct_answer_must_state': 'No stated connection between a second opinion and a lift outage; '
                               'any causal story is invented.',
  'expect_connected': False,
  'kind': 'Cross-category causal'}]

PUBLIC_POSITIVE = [{'id': 'PP1',
  'question': 'Did Patient P2044 register and consent at the front desk and later have an '
              'insurance claim submitted?',
  'shared_identifier': 'Patient P2044',
  'shared_records': ['Patient P2044 registered at the front desk and completed the consent forms',
                     "Patient P2044's insurance claim was submitted after the consultation"],
  'correct_answer_must_state': 'Yes -- both records name Patient P2044.',
  'expect_connected': True},
 {'id': 'PP2',
  'question': 'Did Patient P2063 book a follow-up and also get discharged with home care '
              'instructions?',
  'shared_identifier': 'Patient P2063',
  'shared_records': ['Patient P2063 booked a follow-up appointment after the first visit',
                     'Patient P2063 was discharged with home care instructions'],
  'correct_answer_must_state': 'Yes -- both records name Patient P2063.',
  'expect_connected': True},
 {'id': 'PP3',
  'question': 'Was Prescription RX114 written after the examination and then dispensed by the '
              'pharmacy?',
  'shared_identifier': 'Prescription RX114',
  'shared_records': ['Prescription RX114 was written after the examination',
                     'Prescription RX114 was dispensed by the pharmacy'],
  'correct_answer_must_state': 'Yes -- both records name Prescription RX114.',
  'expect_connected': True}]

print(f"Public pack: {len(PUBLIC_ADVERSARIAL)} adversarial + {len(PUBLIC_POSITIVE)} positive-control questions")

In [ ]:
def structural_check(test):
    """No LLM needed: does the graph connect the two records this question is about?
    Returns (passed, connected, via)."""
    recs = test.get("trap_records") or test.get("shared_records")
    ia, ib = [f"R{rec_texts.index(t)}" for t in recs]
    chk = graph_check(ia, ib)
    connected = not chk["disconnected"]
    return connected == test["expect_connected"], connected, chk["via"]


def run_pack(pack, show_answers=True):
    """Runs baseline and grounded side by side on every question in a pack."""
    rows = []
    for t in pack:
        ok, connected, via = structural_check(t)
        print(f"[{t['id']}] {t['question']}")
        if show_answers:
            print("   Baseline :", baseline_rag_answer(t["question"])["answer"][:350].replace("\n", " "))
            print("   Grounded :", grounded_answer(t["question"])["answer"][:350].replace("\n", " "))
        print(f"   Graph    : {'CONNECTED' if connected else 'NOT connected'}"
              f"{' via ' + ', '.join(via) if via else ''} | expected "
              f"{'connected' if t['expect_connected'] else 'NOT connected'} -> {'PASS' if ok else 'FAIL'}")
        print("   Must say :", t["correct_answer_must_state"][:200])
        print()
        rows.append({"id": t["id"], "graph_pass": ok, "connected": connected})
    print(f"Graph verdict correct on {sum(r['graph_pass'] for r in rows)}/{len(rows)}")
    return rows

### Adversarial questions

In [ ]:
adv_rows = run_pack(PUBLIC_ADVERSARIAL)

### Positive controls: can your agent say *yes*?

In [ ]:
pos_rows = run_pack(PUBLIC_POSITIVE)

## 10. YOUR TASK: explain the outcomes

For every public question, record whether the **baseline fabricated** a connection (read its answer, `True`/`False`), whether your **grounded** answer was correct, and **why**, in terms of your patterns. Then name one failure you still have. Be specific: a short honest note beats a vague claim.

In [ ]:
OUTCOMES = [
    # {"id": "P1", "baseline_fabricated": None, "grounded_correct": None, "explanation": "..."},
    # one entry for each of P1-P8 and PP1-PP3
]

REMAINING_FAILURE = """
TODO: one case, or kind of case, where your system still gets it wrong (or could), and why.
"""

## 11. Submission summary

Run this last. This is what the judges read first.

In [ ]:
import json

def submission_summary():
    print("=== PATTERN REGISTER ===")
    show_register()
    # one machine-readable line the judges' tool reads: keep it
    print("REGISTRY_JSON:", json.dumps([{"name": p["name"], "regex": p["regex"]} for p in PATTERNS]))
    print("DATASET_ID:", DATASET_ID)
    print("\n=== OUTCOMES (public pack) ===")
    print(f"{'ID':<5} {'Baseline fabricated?':<22} {'Grounded correct?':<19} Explanation")
    for o in OUTCOMES:
        print(f"{o['id']:<5} {str(o['baseline_fabricated']):<22} {str(o['grounded_correct']):<19} {o['explanation']}")
    print("\n=== ONE FAILURE WE STILL HAVE ===")
    print(REMAINING_FAILURE)

submission_summary()

## 12. Before you submit

- Restart the runtime and **Run all**: it must work from a fresh state.
- `validate_registry()` reports no problems.
- `OUTCOMES` has an entry for every public question; `REMAINING_FAILURE` is filled in.
- Submit by the deadline announced on Day 1.